# Figure 1 — cfDNA molecular features and correlation to gene expression

**Panels** — (a) normalized sequencing coverage at TSSs for housekeeping
(HK) vs lowly expressed (PAU) genes across the panel-of-normals (mean ± SD,
n = 50); (b) distribution of mean coverage per TSS 10 kb window for the two
gene sets (Mann-Whitney U test); (c) mean per-sample Pearson correlation of
each fragmentomic feature with reference gene-group expression (Esfahani et
al.), for all TSSs vs QC-filtered TSSs.

**Source data** (no coverage arrays, no cluster paths, no network):
- `data/fig1a_coverage_profiles.csv` — per-position mean/SD of the
  normalized aggregate coverage profile per gene set over the PoN.
- `data/fig1b_locus_coverage.csv` — per-sample per-locus mean coverage for
  the two gene sets (after the original per-sample low-coverage mask).
- `data/fig1c_expression_correlations.csv` — per-sample Pearson r per
  feature, for all TSSs and QC TSSs.

Written by `export/export_fig1.py`, which replicates
`src/comp/plot_aggregated_coverage.py` (panels a/b) and
`notebooks/esfahani.ipynb` (panel c). Run top to bottom from `src/figures/`.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from scipy.stats import gaussian_kde, mannwhitneyu

HK_COLOR = (148 / 255, 203 / 255, 236 / 255)
PAU_COLOR = (194 / 255, 106 / 255, 119 / 255)
FEATURE_COLORS = {
    "fslr": (46/255, 37/255, 133/255), "fslr_coverage": (51/255, 117/255, 56/255),
    "griffin_diff": (93/255, 168/255, 153/255), "rcov": (220/255, 205/255, 125/255),
    "mwh": (159/255, 74/255, 150/255),
}
FEATURE_NAMES = {"fslr": "FSLR", "fslr_coverage": "FSLRC", "griffin_diff": "GD",
                 "rcov": "RCOV", "mwh": "MWH"}
FEATURES = list(FEATURE_NAMES)

prof = pd.read_csv("data/fig1a_coverage_profiles.csv")
locus = pd.read_csv("data/fig1b_locus_coverage.csv")
corr = pd.read_csv("data/fig1c_expression_correlations.csv")
N_SAMPLES = corr["sample_index"].nunique()
print(f"PoN samples: {N_SAMPLES}; per-locus rows: {len(locus)}")

## Panel a — coverage profiles at the TSS

In [ ]:
def panel_a(ax):
    for gs, label, color in [("HK", "Housekeeping genes", HK_COLOR),
                             ("PAU", "PAU genes", PAU_COLOR)]:
        d = prof[prof["gene_set"] == gs].sort_values("position")
        mean = d["mean_normalized_coverage"].to_numpy()
        sd = d["sd_normalized_coverage"].to_numpy()
        ax.plot(d["position"], mean, color=color, linewidth=2, alpha=0.9,
                label=f"{label}\n(mean ± SD, n={N_SAMPLES})")
        ax.fill_between(d["position"], mean - sd, mean + sd, color=color, alpha=0.2)
    ax.axvline(x=0, color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax.set_xlabel("Distance from TSS (bp)", fontsize=14)
    ax.set_ylabel("Normalized coverage", fontsize=14)
    ax.tick_params(labelsize=12)
    ax.legend(loc="lower right", fontsize=11, framealpha=0.95)
    ax.grid(True, alpha=0.3, linestyle="--", axis="x")
    ax.set_xlim(-5000, 5000)
    ax.set_xticks([-5000, -2500, 0, 2500, 5000])
    ax.set_xticklabels(["-5kb", "-2.5kb", "TSS", "+2.5kb", "+5kb"])

## Panel b — per-locus mean coverage distributions

In [ ]:
def panel_b(ax):
    hk = locus.loc[locus["gene_set"] == "HK", "mean_coverage"].to_numpy()
    pau = locus.loc[locus["gene_set"] == "PAU", "mean_coverage"].to_numpy()
    n_hk_tss = locus[locus["gene_set"] == "HK"].groupby("sample_id").size().max()
    n_pau_tss = locus[locus["gene_set"] == "PAU"].groupby("sample_id").size().max()

    y_max, means = 0, {}
    x_max = 5
    for label, data, color, n_tss in [("HK genes", hk, HK_COLOR, n_hk_tss),
                                      ("PAU genes", pau, PAU_COLOR, n_pau_tss)]:
        kde = gaussian_kde(data, bw_method="scott")
        x_range = np.linspace(max(data.min(), 0), x_max, 500)
        density = kde(x_range)
        y_max = max(y_max, density.max())
        ax.plot(x_range, density, color=color, linewidth=2,
                label=f"{label} (#TSS={n_tss})")
        ax.fill_between(x_range, density, alpha=0.15, color=color)
        for q_val, q_label in zip(np.percentile(data, [25, 50, 75]), ["Q1", "Median", "Q3"]):
            ax.vlines(q_val, 0, kde(q_val)[0], colors=color, linestyles="dashed",
                      linewidth=1.5 if q_label == "Median" else 1.0, alpha=0.8)
        means[label] = np.mean(data)

    stat, pval = mannwhitneyu(hk, pau, alternative="two-sided")
    p_text = "p < 0.001" if pval < 0.001 else f"p = {pval:.3f}"
    left, right = sorted([means["HK genes"], means["PAU genes"]])
    tick_h, bracket_y = y_max * 0.03, y_max * 1.08
    ax.plot([left, left], [bracket_y - tick_h, bracket_y], color="black",
            linewidth=1.2, clip_on=False)
    ax.plot([left, right], [bracket_y, bracket_y], color="black",
            linewidth=1.2, clip_on=False)
    ax.plot([right, right], [bracket_y - tick_h, bracket_y], color="black",
            linewidth=1.2, clip_on=False)
    ax.text((left + right) / 2, bracket_y * 1.01, p_text, ha="center",
            va="bottom", fontsize=12, fontstyle="italic")

    ax.legend(fontsize=11, framealpha=0.95)
    ax.set_xlabel("Mean coverage per TSS 10k bp window", fontsize=14)
    ax.set_ylabel("Density", fontsize=14)
    ax.tick_params(labelsize=12)
    ax.set_xlim(0, x_max)
    ax.set_ylim(0, bracket_y * 1.15)
    return stat, pval

## Panel c — feature–expression correlations

In [ ]:
def panel_c(ax):
    all_means, all_stds, qc_means, qc_stds, colors, labels = [], [], [], [], [], []
    for feat in FEATURES:
        for sink_m, sink_s, tss_set in [(all_means, all_stds, "all"),
                                        (qc_means, qc_stds, "qc")]:
            v = corr[(corr["feature"] == feat)
                     & (corr["tss_set"] == tss_set)]["pearson_r"].dropna()
            sink_m.append(v.mean())
            sink_s.append(v.std(ddof=0))
        colors.append(FEATURE_COLORS[feat])
        labels.append(FEATURE_NAMES[feat])

    bar_height = 0.35
    positions = np.arange(len(FEATURES)) * 1.0
    ax.barh(positions - bar_height / 2, all_means, bar_height, xerr=all_stds,
            capsize=4, alpha=0.9, color=colors,
            error_kw={"linewidth": 1.5, "ecolor": "black"},
            label=f"All TSSs\n(mean ± SD, n = {N_SAMPLES})",
            edgecolor="black", linewidth=1.5)
    ax.barh(positions + bar_height / 2, qc_means, bar_height, xerr=qc_stds,
            capsize=4, alpha=0.7, color=colors,
            error_kw={"linewidth": 1.5, "ecolor": "black"},
            label=f"QC TSSs\n(mean ± SD, n = {N_SAMPLES})",
            edgecolor="black", linewidth=1.5, hatch="///")

    for means, stds, offset in [(all_means, all_stds, -bar_height / 2),
                                (qc_means, qc_stds, +bar_height / 2)]:
        for mean, std, pos in zip(means, stds, positions):
            x_pos = mean - std - 0.01 if mean > 0 else mean + std + 0.01
            ha = "right" if mean > 0 else "left"
            t = ax.text(x_pos, pos + offset, f"{mean:.3f}", ha=ha, va="center",
                        fontsize=11, fontweight="bold", color="white")
            t.set_path_effects([path_effects.Stroke(linewidth=3, foreground="black"),
                                path_effects.Normal()])

    ax.set_xlabel("Pearson correlation with expression (r)", fontsize=14)
    ax.set_yticks(positions)
    ax.set_yticklabels(labels, fontsize=12)
    ax.grid(True, alpha=0.3, axis="x", linestyle="--")
    ax.axvline(x=0, color="k", linestyle="-", linewidth=0.8)
    ax.legend(fontsize=11, loc="lower right", framealpha=0.9)
    ax.invert_yaxis()
    ax.tick_params(labelsize=12)

## Assemble Figure 1

In [ ]:
fig, (ax_a, ax_b, ax_c) = plt.subplots(1, 3, figsize=(21, 5.2),
                                       gridspec_kw={"width_ratios": [1.2, 1, 1]})
panel_a(ax_a)
stat, pval = panel_b(ax_b)
panel_c(ax_c)
for ax, letter in zip((ax_a, ax_b, ax_c), "abc"):
    ax.text(-0.1, 1.06, letter, transform=ax.transAxes,
            fontsize=18, fontweight="bold", va="top")
plt.tight_layout()
os.makedirs("output", exist_ok=True)
fig.savefig("output/fig1.png", dpi=300, bbox_inches="tight")
fig.savefig("output/fig1.pdf", bbox_inches="tight")
plt.show()

## Checks against the manuscript

Fig 1b: the HK vs PAU difference is significant (MWU p < 0.001). Fig 1c
reference values (read off the submitted panel): FSLR 0.713 / 0.735,
FSLRC ~0.708 / ~0.735, GD ~0.706 / ~0.745, RCOV and MWH negative, with the
QC set strengthening every feature's correlation.

In [ ]:
print(f"Panel b MWU: U={stat:.0f}, p={pval:.3g}  (manuscript: p < 0.001)")
assert pval < 0.001

print("\nPanel c (mean ± SD over samples):")
for feat in FEATURES:
    vals = {}
    for tss_set in ["all", "qc"]:
        v = corr[(corr["feature"] == feat) & (corr["tss_set"] == tss_set)]["pearson_r"].dropna()
        vals[tss_set] = (v.mean(), v.std(ddof=0))
    print(f"  {FEATURE_NAMES[feat]:<6} all {vals['all'][0]:+.3f} ± {vals['all'][1]:.3f}   "
          f"qc {vals['qc'][0]:+.3f} ± {vals['qc'][1]:.3f}")
    if feat in ("fslr", "fslr_coverage", "griffin_diff"):
        assert vals["qc"][0] > vals["all"][0] > 0.5, f"{feat}: unexpected correlation pattern"
    else:
        assert vals["all"][0] < 0, f"{feat}: expected negative correlation"